# 02 · RNN factor model — default configuration  *(branch `exp/02-leakage-fixes`)*

Trains the notebook's `EnhancedRnnRegNet` (LSTM+GRU hybrid with attention) with **its own hyperparameters**,
but with both look-ahead paths closed (`CausalEnhancedRnnRegNet`) and with correctly aligned validation/test targets.
Writes the 8 RNN factors for every bar of every split — the baseline agent (03) uses them.

Runtime: the notebook's budget is ~28k steps; on a T4 expect roughly 30–90 min (early stopping usually ends it sooner).
Set `MAX_MINUTES` to cap it. Output: `thesis_rl/rnn/default/` (`best_model.pt`, `factors_*.npy`, `metrics.json`, `history.json`).

In [ ]:
# ---- setup: Drive, code, packages (identical in every notebook) -------------------------
BRANCH = "exp/02-leakage-fixes"                                  # the branch this notebook belongs to
REPO = "https://github.com/taimoor-ahmed-1/FinAI-CryptoRaiders.git"
DRIVE_ROOT = "/content/drive/MyDrive/thesis_rl"      # data in, every output out - survives disconnects

import os, sys, subprocess
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/FinAI-CryptoRaiders"
    if not os.path.exists(REPO_DIR):
        # sparse clone: only experiments/ is checked out - the repo's data folders are never downloaded
        subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "--branch", BRANCH, REPO, REPO_DIR], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "sparse-checkout", "set", "--no-cone", "/experiments/"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/experiments/requirements-colab.txt"], check=True)
    EXP = f"{REPO_DIR}/experiments"
else:                                                # local run from a checkout: notebooks/ -> experiments/
    EXP = os.path.abspath("..")
    DRIVE_ROOT = os.environ.get("THESIS_RL_ROOT", os.path.abspath("../../thesis_rl_outputs"))
sys.path.insert(0, EXP)
os.chdir(EXP)

import torch
from rltrade import paths, seeding
L = paths.layout(DRIVE_ROOT)
print("code:", BRANCH, seeding.git_commit(EXP), "| device:", "cuda - " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
if not torch.cuda.is_available(): print('WARNING: no GPU - Runtime > Change runtime type > T4 GPU')

In [ ]:
seeding.record_environment(L.report / "environment", "02_rnn_train")   # exact environment of this run (pip freeze, CUDA, GPU)

In [ ]:
from rltrade.rnn.train import RnnConfig, train_rnn
ARCH, SEED = "enhanced", 0
MAX_MINUTES = 0                     # 0 = no cap (notebook budget + early stopping)
if not (L.rnn_default / "metrics.json").exists():
    metrics = train_rnn(RnnConfig(arch=ARCH, seed=SEED, max_minutes=MAX_MINUTES), L.prepared, L.rnn_default)
else:
    print("already trained - delete", L.rnn_default, "to retrain")

## Results
Validation loss must **not** fall monotonically forever — that pattern was the symptom of the leak (brief §3.3). IC = correlation between prediction and label per horizon; direction accuracy ignores zero labels.

In [ ]:
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt
m = json.loads((L.rnn_default / "metrics.json").read_text())
h = pd.DataFrame(json.loads((L.rnn_default / "history.json").read_text()))
print(m["stop_reason"], "| steps", m["steps_run"], "|", m["minutes"], "min")
print(pd.DataFrame({s: {k: m[s][k] for k in ("mse", "ic_mean", "dir_acc_mean")} for s in ("train", "val", "test")}).round(4))
horizons = [10, 20, 30, 60, 80, 100, 200, 400]
print(pd.DataFrame({s + "_ic": m[s]["ic"] for s in ("val", "test")}, index=[f"{w} bars" for w in horizons]).round(3))
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(h.step, h.train_loss, lw=.8, alpha=.6, label="train (noisy minibatch)"); ax.plot(h.step, h.val_loss, lw=1.6, label="validation")
b = h.val_loss.idxmin(); ax.axvline(h.step[b], ls="--", c="gray"); ax.set_xlabel("step"); ax.set_ylabel("MSE"); ax.legend(); ax.grid(alpha=.3)
monotone = bool((h.val_loss.diff().dropna() <= 0).all())
ax.set_title(f"Causal model - best val at step {h.step[b]}; val loss {'monotone (!)' if monotone else 'not monotone'}")
fig.savefig(L.report / "rnn_default_loss.png", dpi=150); plt.show()

## Optional · leakage demonstration
Trains the **original, leaky** architecture (`enhanced_legacy`) for a short budget on the same (correct) targets.
If it reaches a much lower validation loss than the causal model, that gap is information read from the future.
Record the outcome in `EXPERIMENTS.md` / `LEAKAGE_AUDIT.md`.

In [ ]:
RUN_LEAK_DEMO = False
if RUN_LEAK_DEMO:
    demo = {}
    for arch in ("enhanced", "enhanced_legacy"):
        d = L.root / "rnn" / "leak_demo" / arch
        demo[arch] = train_rnn(RnnConfig(arch=arch, seed=0, max_minutes=20), L.prepared, d, save_factors=False)
    print(pd.DataFrame({a: {"val_mse": x["val"]["mse"], "val_ic": x["val"]["ic_mean"], "test_ic": x["test"]["ic_mean"]}
                        for a, x in demo.items()}).round(4))